In [28]:
import os
import logging
import argparse
import pandas as pd
from dotenv import load_dotenv
from neo4j_handler import Neo4jHandler

# Import Splink components using the confirmed syntax
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets

# Configure logging.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables from the .env file.
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not (NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD):
    raise ValueError("Please set NEO4J_URI, NEO4J_USER, and NEO4J_PASSWORD in your .env file")

handler = Neo4jHandler(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

logger.info("Initializing DuckDB backend for Splink...")
db_api = DuckDBAPI()  # Create a DuckDB backend instance
from splink.exploratory import profile_columns

INFO:neo4j_handler:Connected to Neo4j at bolt://localhost:7687 as user neo4j
INFO:__main__:Initializing DuckDB backend for Splink...


In [29]:
def fetch_identities(handler):
    """
    Fetch all Identity nodes from the Neo4j database and return them as a list of dictionaries.
    Expected columns: id, full_name, email_address, zip_code, phone_number.
    """
    query = """
    MATCH (i:Identity)
    RETURN distinct i
    """
    result = handler.execute_read(query)
    # Consume the result within the transaction scope.
    records = [record.data() for record in result]
    return records


identities = fetch_identities(handler)

In [30]:
data = [identity["i"] for identity in identities]  
identities_df = pd.DataFrame(data)


identities_df.head()



,email_address,wcc_component,last_name,phone_number,id,full_address,first_name
0,moreyHolmes@gmail.com,0,Holmes,2806901865,895a9541-fa30-4d6d-836b-008ecff40135,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
1,moreyHolmes@gmail.com,0,Holmes,2806901865,642d0174-c5d5-4666-854c-1d9f790e32c4,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
2,moreyHolmes@gmail.com,0,Holmes,2806901865,297420aa-b8c1-4274-8f91-e5cf0619a421,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
3,moreyHolmes@gmail.com,0,Holmes,2806901865,e3c69cc4-62cc-4bea-be66-b3079e5d37f7,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
4,NaN,0,qolmes,2806901865,86b0cfd4-5a10-477e-ba48-6eaab01b21a0,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice


In [31]:
identities_df = identities_df.rename(columns={"id": "unique_id"})
identities_df.head()

,email_address,wcc_component,last_name,phone_number,unique_id,full_address,first_name
0,moreyHolmes@gmail.com,0,Holmes,2806901865,895a9541-fa30-4d6d-836b-008ecff40135,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
1,moreyHolmes@gmail.com,0,Holmes,2806901865,642d0174-c5d5-4666-854c-1d9f790e32c4,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
2,moreyHolmes@gmail.com,0,Holmes,2806901865,297420aa-b8c1-4274-8f91-e5cf0619a421,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
3,moreyHolmes@gmail.com,0,Holmes,2806901865,e3c69cc4-62cc-4bea-be66-b3079e5d37f7,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
4,NaN,0,qolmes,2806901865,86b0cfd4-5a10-477e-ba48-6eaab01b21a0,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice


In [32]:

profile_columns(identities_df, db_api, column_expressions=["first_name", "last_name", "email_address", "phone_number",])

alt.VConcatChart(...)

In [33]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)

blocking_rules = [
    block_on("wcc_component"),
    block_on("substr(last_name, 1, 3)", "substr(first_name, 1, 2)"),

    
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=identities_df,
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

In [43]:


# Add a source_dataset column to the DataFrame.
identities_df["source_dataset"] = "dataset_1"  # Assign a default value for all records.

logger.info(f"Fetched {len(identities_df)} identity records.")

# Create Splink settings using the new syntax.
# Here we use our existing columns:
# - NameComparison on full_name
# - EmailComparison on email_address
# - ExactMatch on zip_code and phone_number (with term frequency adjustments for zip_code)
settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        # cl.NameComparison("full_name"),
        # cl.EmailComparison("email_address"),

        # cl.ForenameSurnameComparison(
        #     "first_name",
        #     "last_name",
        #     forename_surname_concat_col_name="first_name_last_name_concat",
        # ),
        cl.NameComparison("first_name").configure(term_frequency_adjustments=True),
        cl.NameComparison("last_name").configure(term_frequency_adjustments=True),
        cl.LevenshteinAtThresholds("email_address"),
        cl.ExactMatch("phone_number"),
        cl.LevenshteinAtThresholds("full_address")
    ],
    #     cl.LevenshteinAtThresholds("full_name").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("email_address").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("zip_code").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("phone_number").configure(term_frequency_adjustments=True),
    # ],
    retain_intermediate_calculation_columns=True,
    blocking_rules_to_generate_predictions=blocking_rules,
)


# identities_df["first_name_last_name_concat"] = identities_df["first_name"] + " " + identities_df["last_name"]

logger.info("Initializing Splink Linker...")
# Initialize the generic Linker with the DataFrame, settings, and DuckDB backend.
linker = Linker(identities_df, settings, db_api=db_api)



INFO:__main__:Fetched 6352 identity records.
INFO:__main__:Initializing Splink Linker...


In [44]:
# -------------------------
# Training steps:
# Estimate the probability that two random records match.
linker.training.estimate_probability_two_random_records_match(
    [block_on("wcc_component")],
    recall=0.8,
)

INFO:splink.internals.linker_components.training:Probability two random records match is estimated to be  0.0105.
This means that amongst all possible pairwise record comparisons, one in 95.25 are expected to match.  With 20,170,776 total possible comparisons, we expect a total of around 211,761.25 matching pairs


In [45]:
# Estimate u probabilities using random sampling.
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)


INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - last_name (no m values are trained).
    - email_address (no m values are trained).
    - phone_number (no m values are trained).
    - full_address (no m values are trained).


In [46]:

training_blocking_rule = block_on("wcc_component")
trainin_sesion_wcc= (
    linker.training.estimate_parameters_using_expectation_maximisation(
        training_blocking_rule
    )
)

INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
l."wcc_component" = r."wcc_component"

Parameter estimates will be made for the following comparison(s):
    - first_name
    - last_name
    - email_address
    - phone_number
    - full_address

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was 0.166 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was -0.00392 in the m_probability of email_address, level `Exact match on email_address`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params was -0.000258 in the m_probability of first_name, level `Exact 

In [47]:

# training_blocking_rule = block_on("email_address")
# trainin_sesion_phone= (
#     linker.training.estimate_parameters_using_expectation_maximisation(
#         training_blocking_rule, estimate_without_term_frequencies=True
#     )
# )

In [48]:
linker.visualisations.match_weights_chart()

alt.VConcatChart(...)

In [64]:
linker.visualisations.parameter_estimate_comparisons_chart()


alt.Chart(...)

In [49]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

In [50]:
df_predict = linker.inference.predict()
df_e = df_predict.as_pandas_dataframe()
# df_e = df_e[abs(df_e["match_probability"]) < 1]
df_e = df_e[abs(df_e["match_probability"]) > .5]

# df_e = df_e[df_e["match_weig"] > 0.1]

INFO:splink.internals.linker_components.inference:Blocking time: 0.08 seconds
INFO:splink.internals.linker_components.inference:Predict time: 3.15 seconds


In [60]:
records_to_plot = df_e.head(20).to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)

alt.LayerChart(...)

In [ ]:
# what are m and u for this model


In [53]:
# df_clusters = clusters.as_pandas_dataframe()
# df_clusters.head()

In [54]:
# # write the clusters to neo4j
# rows = df_clusters.to_dict("records")

# # Cypher query to update nodes: it unwinds each row and sets the cluster_id property.
# update_query = """
# UNWIND $rows as row
# MATCH (n {id: row.id})
# SET n.cluster_id = row.cluster_id
# RETURN count(n) AS updated_count
# """

# # Use your Neo4jHandler's execute_write method to run the update.
# handler.execute_write(update_query, parameters={"rows": rows})

In [56]:
#show all keys in df
df_e.keys()

Index(['match_weight', 'match_probability', 'unique_id_l', 'unique_id_r',
       'first_name_l', 'first_name_r', 'gamma_first_name', 'tf_first_name_l',
       'tf_first_name_r', 'bf_first_name', 'bf_tf_adj_first_name',
       'last_name_l', 'last_name_r', 'gamma_last_name', 'tf_last_name_l',
       'tf_last_name_r', 'bf_last_name', 'bf_tf_adj_last_name',
       'email_address_l', 'email_address_r', 'gamma_email_address',
       'bf_email_address', 'phone_number_l', 'phone_number_r',
       'gamma_phone_number', 'bf_phone_number', 'full_address_l',
       'full_address_r', 'gamma_full_address', 'bf_full_address',
       'wcc_component_l', 'wcc_component_r', 'match_key'],
      dtype='object')

In [69]:
df_e.head(1)

,match_weight,match_probability,unique_id_l,unique_id_r,first_name_l,first_name_r,gamma_first_name,tf_first_name_l,tf_first_name_r,bf_first_name,...,gamma_full_address,bf_full_address,wcc_component_l,wcc_component_r,match_key,w_first_name,w_last_name,w_email_address,w_phone_number,w_full_address
0,20.852883,0.999999,34bad150-1cc1-4958-bb15-4ba22cbd8368,3d6713e0-bfb1-44b4-9a8d-8820fe0fac67,Yvonne,Yvonne,4,0.001259,0.001259,325.737844,...,3,676.605483,475,475,0,8.35,7.82,0.0,0.0,9.4


In [68]:
import math
# create helper funtion for getting log2 of bf
def get_weight(bf):
    return 0 if bf == 0 else round(math.log2(bf), 2)

#calculate weight of each column in pandas
df_e["w_first_name"] = df_e["bf_first_name"].apply(get_weight)
df_e["w_last_name"] = df_e["bf_last_name"].apply(get_weight)    
df_e["w_email_address"] = df_e["bf_email_address"].apply(get_weight)
df_e["w_phone_number"] = df_e["bf_phone_number"].apply(get_weight)
df_e["w_full_address"] = df_e["bf_full_address"].apply(get_weight)


In [70]:



rows = [
    {
        "unique_id_l": row["unique_id_l"],
        "unique_id_r": row["unique_id_r"],
        "rel_properties": {
            "probability": row["match_probability"],
            "match_key": row["match_key"],
            "match_weight": row["match_weight"],
            "first_name_match_weight": row["w_first_name"],
            "last_name_match_weight": row["w_last_name"],
            "email_address_match_weight": row["w_email_address"],
            "phone_number_match_weight": row["w_phone_number"],
            
        },
    }
    for _, row in df_e.iterrows()
]

# Cypher query to update nodes
update_query = """
    UNWIND $rows as row
    MATCH (a:Identity {id: row.unique_id_l})
    MATCH (b:Identity {id: row.unique_id_r})
    WHERE NOT EXISTS((a)<-[:contains_identity]-()-[:contains_identity]->(b))
    MERGE (a)-[r:SPLINK]-(b)
    SET r += row.rel_properties
    RETURN a, b, r
"""

# Execute the query
handler.execute_write(update_query, parameters={"rows": rows})

In [ ]:
# import json

# for _, row in df_e.iterrows():
#     handler.create_relationship(
#         source_label="Identity",
#         source_key="id",
#         source_value=row["unique_id_l"],
#         target_label="Identity",
#         target_key="id",
#         target_value=row["unique_id_r"],
#         relationship_type="MATCHED",
#         rel_properties={
#             "probability": row["match_probability"],
#             "data": json.dumps(row.to_dict())  # Serialize the row as a JSON string
#         }
#     )

In [84]:


# Step 1: Fetch identityIds.
fetch_ids_query = """
MATCH (n:Identity)
RETURN collect(id(n)) AS identityIds
"""

identity_ids_result = handler.execute_read(fetch_ids_query)
identity_ids = identity_ids_result[0]["identityIds"]


In [131]:
drop_query = """
    CALL gds.graph.exists('leidenGraph') YIELD exists
    WITH exists
    WHERE exists
    CALL gds.graph.drop('leidenGraph') YIELD graphName
    RETURN graphName
    """
handler.execute_write(drop_query)
projection_query = """
MATCH (source)-[r:SPLINK]->(target)
WITH gds.graph.project(
  'leidenGraph',
  source,
  target,
  {relationshipType: type(r), relationshipProperties: r { .match_weight }},
  {undirectedRelationshipTypes: ['*']}
) as g

RETURN g.graphName AS graph, g.nodeCount AS nodes, g.relationshipCount AS rels
"""
with handler.driver.session() as session:
    result = session.run(projection_query)
    for record in result:
        print(record)

<Record graph='leidenGraph' nodes=2651 rels=15162>


In [149]:
leiden_query = """
    CALL gds.leiden.write('leidenGraph', { writeProperty: 'communityId', relationshipWeightProperty: 'match_weight'})
    YIELD communityCount, communityDistribution
    """
with handler.driver.session() as session:
    leiden_results = session.run(leiden_query)
    results = [record.data() for record in leiden_results]
    # logger.info("Leiden results (sample):")
    # for r in results[:10]:
    #     logger.info(r)

In [150]:
results

[{'communityCount': 275,
  'communityDistribution': {'min': 5,
   'p5': 5,
   'max': 22,
   'p999': 22,
   'p99': 21,
   'p1': 5,
   'p10': 5,
   'p90': 15,
   'p50': 9,
   'p25': 7,
   'p75': 12,
   'p95': 17,
   'mean': 9.64}}]